# Country shares for each category.


In [1]:
from __future__ import annotations
from pathlib import Path
import pandas as pd


# ============================================================
# Step 1. File names
# ============================================================


CLASSIFIED_INPUT = Path("step1_indicator_monetary_classification.csv")
WORLDBANK_CLASSIFICATION = Path("worldbank_classification.csv")

COUNTRY_OUTPUT = Path("step2_country_category_shares.csv")
REGION_OUTPUT = Path("step2_region_category_average_shares.csv")
INCOME_OUTPUT = Path("step2_income_category_average_shares.csv")
LONG_OUTPUT = Path("step2_region_income_category_average_shares_long.csv")


# ============================================================
# Step 2. Category columns from Step 1
# ============================================================

CATEGORIES = [
    "Physical stock",
    "Monetary stock",
    "Physical flow",
    "Monetary flow",
    "Action_kept",
    "Policy Guide_kept",
    "HPS_monetary_only",
    "DDP_monetary_only",
    "MS-NMS_monetary_only",
    "ADE_monetary_only",
]


# ============================================================
# Step 3. Validate input files and columns
# ============================================================

def validate_inputs(df: pd.DataFrame, countries: pd.DataFrame) -> list[str]:
    missing_categories = [col for col in CATEGORIES if col not in df.columns]

    if missing_categories:
        raise ValueError(f"Missing Step 1 category columns: {missing_categories}")

    required_wb_columns = ["Country", "Region", "Income"]
    missing_wb_columns = [col for col in required_wb_columns if col not in countries.columns]

    if missing_wb_columns:
        raise ValueError(f"Missing worldbank_classification columns: {missing_wb_columns}")

    country_columns = countries["Country"].dropna().tolist()
    missing_country_columns = [col for col in country_columns if col not in df.columns]

    if missing_country_columns:
        raise ValueError(
            "Countries in worldbank_classification are missing from the availability file: "
            f"{missing_country_columns}"
        )

    return country_columns


# ============================================================
# Step 4. Calculate country-level shares
# ============================================================

def calculate_country_shares(
    df: pd.DataFrame,
    countries: pd.DataFrame,
    country_columns: list[str],
) -> tuple[pd.DataFrame, dict[str, int]]:

    denominators = {category: int(df[category].sum()) for category in CATEGORIES}
    rows = []

    for country in country_columns:
        row = {"Country": country}

        for category in CATEGORIES:
            denominator = denominators[category]

            if denominator == 0:
                row[category] = pd.NA
                row[f"{category}_available_count"] = 0
                row[f"{category}_denominator"] = 0
                continue

            category_mask = df[category] == 1
            available_count = int(
                df.loc[category_mask, country].fillna(0).astype(int).sum()
            )

            row[category] = round(available_count / denominator * 100, 2)
            row[f"{category}_available_count"] = available_count
            row[f"{category}_denominator"] = denominator

        rows.append(row)

    country_shares = pd.DataFrame(rows)

    country_shares = country_shares.merge(
        countries[["Country", "Region", "Income"]],
        on="Country",
        how="left",
    )

    leading_cols = ["Country", "Region", "Income"]
    other_cols = [col for col in country_shares.columns if col not in leading_cols]

    return country_shares[leading_cols + other_cols], denominators


# ============================================================
# Step 5. Calculate regional and income-group averages
# ============================================================

def average_by_group(country_shares: pd.DataFrame, group_column: str) -> pd.DataFrame:
    grouped = (
        country_shares.groupby(group_column, dropna=False)[CATEGORIES]
        .mean()
        .round(2)
        .reset_index()
    )

    grouped.insert(
        1,
        "country_count",
        country_shares.groupby(group_column, dropna=False)["Country"].nunique().values,
    )

    return grouped


# ============================================================
# Step 6. Create long-format output for figures
# ============================================================

def build_long_output(region_avg: pd.DataFrame, income_avg: pd.DataFrame) -> pd.DataFrame:
    region_long = region_avg.melt(
        id_vars=["Region", "country_count"],
        value_vars=CATEGORIES,
        var_name="category",
        value_name="average_country_share_percent",
    ).rename(columns={"Region": "group"})

    region_long.insert(0, "group_type", "Region")

    income_long = income_avg.melt(
        id_vars=["Income", "country_count"],
        value_vars=CATEGORIES,
        var_name="category",
        value_name="average_country_share_percent",
    ).rename(columns={"Income": "group"})

    income_long.insert(0, "group_type", "Income")

    return pd.concat([region_long, income_long], ignore_index=True)


# ============================================================
# Step 7. Run analysis
# ============================================================

df = pd.read_csv(CLASSIFIED_INPUT)
countries = pd.read_csv(WORLDBANK_CLASSIFICATION)

country_columns = validate_inputs(df, countries)

country_shares, denominators = calculate_country_shares(
    df,
    countries,
    country_columns,
)

region_avg = average_by_group(country_shares, "Region")
income_avg = average_by_group(country_shares, "Income")
long_output = build_long_output(region_avg, income_avg)

country_shares.to_csv(COUNTRY_OUTPUT, index=False, encoding="utf-8-sig")
region_avg.to_csv(REGION_OUTPUT, index=False, encoding="utf-8-sig")
income_avg.to_csv(INCOME_OUTPUT, index=False, encoding="utf-8-sig")
long_output.to_csv(LONG_OUTPUT, index=False, encoding="utf-8-sig")

print("Step 2 country, region, and income category shares complete")
print(f"Countries: {len(country_columns):,}")

print("Category denominators:")
for category, denominator in denominators.items():
    print(f"  {category}: {denominator}")

print()
print(f"Country output: {COUNTRY_OUTPUT}")
print(f"Region output: {REGION_OUTPUT}")
print(f"Income output: {INCOME_OUTPUT}")
print(f"Long output: {LONG_OUTPUT}")

Step 2 country, region, and income category shares complete
Countries: 218
Category denominators:
  Physical stock: 52
  Monetary stock: 91
  Physical flow: 202
  Monetary flow: 221
  Action_kept: 14
  Policy Guide_kept: 740
  HPS_monetary_only: 30
  DDP_monetary_only: 188
  MS-NMS_monetary_only: 210
  ADE_monetary_only: 32

Country output: step2_country_category_shares.csv
Region output: step2_region_category_average_shares.csv
Income output: step2_income_category_average_shares.csv
Long output: step2_region_income_category_average_shares_long.csv
